# Data Exploration with R

**Learning Goal:** Use Claude Code to explore Bolivia's sustainable development data using R.

This notebook demonstrates:
- Loading data using project configuration
- Basic data inspection
- Summary statistics
- Identifying patterns and relationships

This is the R equivalent of `01_data_exploration.ipynb` (Python version).

## Setup

First, we load our project configuration and required libraries.

In [ ]:
# Load project configuration
source('../config.R')

# Set seeds for reproducibility
set_seeds()

# Load required libraries
library(tidyverse)

cat("Data directory:", DATA_DIR, "\n")

## Load the Datasets

We have 4 datasets that can be joined via `asdf_id`:
1. **regionNames** - Municipality identifiers
2. **sdg** - SDG index scores (0-100)
3. **sdgVariables** - Detailed SDG indicators
4. **ntl** - Night-time lights (economic proxy)

In [ ]:
# Load all datasets
regions <- read_csv(file.path(DATA_DIR, 'regionNames', 'regionNames.csv'), show_col_types = FALSE)
sdg <- read_csv(file.path(DATA_DIR, 'sdg', 'sdg.csv'), show_col_types = FALSE)
sdg_vars <- read_csv(file.path(DATA_DIR, 'sdgVariables', 'sdgVariables.csv'), show_col_types = FALSE)
ntl <- read_csv(file.path(DATA_DIR, 'ntl', 'ln_NTLpc.csv'), show_col_types = FALSE)

cat("Regions:", dim(regions), "\n")
cat("SDG indices:", dim(sdg), "\n")
cat("SDG variables:", dim(sdg_vars), "\n")
cat("Night-time lights:", dim(ntl), "\n")

## Inspect the Data

Let's look at each dataset's structure.

In [ ]:
# Region names - administrative metadata
cat("=== REGION NAMES ===", "\n")
head(regions)
cat("\nColumns:", names(regions), "\n")

In [ ]:
# SDG indices - composite scores
cat("=== SDG INDICES ===", "\n")
head(sdg)
cat("\nColumns:", names(sdg), "\n")

In [ ]:
# Night-time lights - economic activity proxy
cat("=== NIGHT-TIME LIGHTS ===", "\n")
head(ntl)
cat("\nColumns:", names(ntl), "\n")

## Merge Datasets

Create a combined dataset for analysis using dplyr joins.

In [ ]:
# Merge all datasets on asdf_id
df <- regions %>%
  left_join(sdg, by = 'asdf_id') %>%
  left_join(ntl, by = 'asdf_id')

cat("Combined dataset:", dim(df), "\n")
cat("\nColumns:", names(df), "\n")

## Summary Statistics

Explore the distribution of SDG indices across Bolivia's municipalities.

In [ ]:
# SDG index columns
sdg_cols <- names(df)[str_starts(names(df), 'index_sdg')]
cat("SDG indices:", sdg_cols, "\n\n")

# Summary statistics
df %>%
  select(all_of(sdg_cols)) %>%
  summary()

In [ ]:
# Overall development index (IMDS)
cat("=== Municipal Sustainable Development Index (IMDS) ===", "\n")
summary(df$imds)

cat("\nTop 5 municipalities:\n")
df %>%
  select(mun, dep, imds) %>%
  arrange(desc(imds)) %>%
  head(5)

In [ ]:
# Department-level summary
cat("=== IMDS by Department ===", "\n")
df %>%
  group_by(dep) %>%
  summarise(
    mean = mean(imds, na.rm = TRUE),
    sd = sd(imds, na.rm = TRUE),
    min = min(imds, na.rm = TRUE),
    max = max(imds, na.rm = TRUE),
    count = n()
  ) %>%
  arrange(desc(mean))

## Quick Visualizations

Using ggplot2 for elegant visualizations.

In [ ]:
# Set plot size for Jupyter
options(repr.plot.width = 12, repr.plot.height = 5)

# Distribution of IMDS - Histogram
p1 <- ggplot(df, aes(x = imds)) +
  geom_histogram(bins = 30, fill = 'steelblue', color = 'black', alpha = 0.7) +
  labs(
    title = 'Distribution of Sustainable Development Index',
    x = 'IMDS Score',
    y = 'Number of Municipalities'
  ) +
  theme_minimal()

# Boxplot by department
p2 <- ggplot(df, aes(x = reorder(dep, imds, FUN = median), y = imds)) +
  geom_boxplot(fill = 'steelblue', alpha = 0.7) +
  labs(
    title = 'IMDS by Department',
    x = 'Department',
    y = 'IMDS Score'
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

# Combine plots side by side
library(gridExtra)
grid.arrange(p1, p2, ncol = 2)

In [ ]:
# Save the plot
ggsave(
  filename = file.path(OUTPUT_DIR, 'imds_distribution_R.png'),
  plot = grid.arrange(p1, p2, ncol = 2),
  width = 12,
  height = 5,
  dpi = 150
)
cat("Plot saved to:", file.path(OUTPUT_DIR, 'imds_distribution_R.png'), "\n")

## Your Turn!

Try asking Claude Code to help you with:
- "Show me the correlation between SDG indices using corrplot"
- "Which municipalities have the lowest SDG1 (poverty) scores?"
- "Plot night-time lights over time for a specific department"
- "Create a faceted plot comparing SDG indices across departments"

In [ ]:
# Space for your exploration
